# Fourier introdutório

Passagem didática do domínio espacial ao domínio da frequência com DFT, magnitude, fase, reconstrução e máscaras ideais circulares.

## Objetivos

- interpretar o espectro complexo centralizado;
- separar magnitude, fase e visualização;
- reconstruir uma imagem com IDFT;
- aplicar passa-baixa e passa-alta ideais;
- comparar conceitualmente filtragem espacial e em frequência.

## 1. Instalação e imports

A instalação commitada usa a branch principal do repositório.

In [ ]:
%pip install -q "git+https://github.com/tfvieira/dip-2026-2.git"

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from dip_toolkit import download_course_image
from dip_toolkit.modules.feature_extractor import FeatureExtractor
from dip_toolkit.modules.fourier_transformer import FourierTransformer
from dip_toolkit.modules.image_loader import ImageLoader
from dip_toolkit.modules.image_preprocessor import ImagePreprocessor

transformer = FourierTransformer()
loader = ImageLoader()
spatial_filters = ImagePreprocessor()
features = FeatureExtractor()

## 2. Espaço x frequência

No domínio espacial observamos pixels e vizinhanças. A DFT descreve a imagem como componentes de frequência. Variações lentas aparecem perto do centro do espectro centralizado; variações rápidas e detalhes aparecem mais afastados.

## 3. DFT de uma imagem sintética

In [ ]:
synthetic = np.zeros((64, 64), dtype=np.float64)
synthetic[16:48, 16:48] = 255.0
spectrum = transformer.dft(synthetic)

print(synthetic.shape, synthetic.dtype)
print(spectrum.shape, spectrum.dtype, np.iscomplexobj(spectrum))

## 4. Centralização do espectro

O contrato público é único: dft() devolve um array complexo 2D, de mesmo shape, já centralizado por fftshift. O zero de frequência (DC) está em (altura // 2, largura // 2). idft() recebe esse formato e aplica ifftshift internamente.

In [ ]:
center = (spectrum.shape[0] // 2, spectrum.shape[1] // 2)
print("Centro:", center)
print("Componente DC:", spectrum[center])

## 5. Magnitude e fase

magnitude() retorna apenas abs(spectrum). A compressão log1p abaixo existe somente para visualização e evita log(0).

In [ ]:
magnitude = transformer.magnitude(spectrum)
phase = transformer.phase(spectrum)
display_magnitude = np.log1p(magnitude)

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(synthetic, cmap="gray")
axes[0].set_title("Imagem sintética")
axes[1].imshow(display_magnitude, cmap="gray")
axes[1].set_title("log1p(magnitude)")
axes[2].imshow(phase, cmap="twilight")
axes[2].set_title("Fase (rad)")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 6. Reconstrução com IDFT

A IDFT retorna float64 e preserva pequenas diferenças numéricas. Clipping e conversão são decisões exclusivas da exibição.

In [ ]:
reconstructed = transformer.idft(spectrum)
display_reconstructed = np.clip(reconstructed, 0, 255).astype(np.uint8)
print("Erro máximo:", np.max(np.abs(reconstructed - synthetic)))

figure, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(synthetic, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(display_reconstructed, cmap="gray")
axes[1].set_title("Reconstruída")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 7. Passa-baixa ideal

A máscara vale 1 dentro do disco central, incluindo sua borda, e 0 fora. O cutoff é um raio em bins de frequência.

In [ ]:
cutoff = 8.0
low_mask = transformer.ideal_low_pass_mask(spectrum.shape, cutoff)
low_spectrum = transformer.apply_mask(spectrum, low_mask)
low_image = transformer.idft(low_spectrum)

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(low_mask, cmap="gray")
axes[0].set_title("Máscara passa-baixa")
axes[1].imshow(np.log1p(transformer.magnitude(low_spectrum)), cmap="gray")
axes[1].set_title("Espectro filtrado")
axes[2].imshow(np.clip(low_image, 0, 255), cmap="gray")
axes[2].set_title("Reconstrução")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 8. Passa-alta ideal

A passa-alta é exatamente 1 - low_mask: remove o disco central e preserva frequências mais afastadas. A reconstrução pode conter valores negativos, por isso seu cálculo não é convertido para uint8.

In [ ]:
high_mask = transformer.ideal_high_pass_mask(spectrum.shape, cutoff)
high_spectrum = transformer.apply_mask(spectrum, high_mask)
high_image = transformer.idft(high_spectrum)
high_limit = np.max(np.abs(high_image))

figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(high_mask, cmap="gray")
axes[0].set_title("Máscara passa-alta")
axes[1].imshow(np.log1p(transformer.magnitude(high_spectrum)), cmap="gray")
axes[1].set_title("Espectro filtrado")
axes[2].imshow(high_image, cmap="gray", vmin=-high_limit, vmax=high_limit)
axes[2].set_title("Reconstrução assinada")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 9. Imagem real

A imagem é obtida por download_course_image(...) e carregada em grayscale pelo ImageLoader; não usamos cv.imread diretamente.

In [ ]:
image_path = download_course_image("cameraman_original.png")
real_image = loader.load_image(image_path, flags=cv.IMREAD_GRAYSCALE)
real_spectrum = transformer.dft(real_image)
real_cutoff = min(real_image.shape) * 0.08

real_low_mask = transformer.ideal_low_pass_mask(real_spectrum.shape, real_cutoff)
real_high_mask = transformer.ideal_high_pass_mask(real_spectrum.shape, real_cutoff)
real_low = transformer.idft(transformer.apply_mask(real_spectrum, real_low_mask))
real_high = transformer.idft(transformer.apply_mask(real_spectrum, real_high_mask))

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(real_image, cmap="gray")
axes[0].set_title("Original")
axes[1].imshow(np.log1p(transformer.magnitude(real_spectrum)), cmap="gray")
axes[1].set_title("Magnitude (log1p)")
axes[2].imshow(np.clip(real_low, 0, 255), cmap="gray")
axes[2].set_title("Passa-baixa")
real_high_limit = np.max(np.abs(real_high))
axes[3].imshow(real_high, cmap="gray", vmin=-real_high_limit, vmax=real_high_limit)
axes[3].set_title("Passa-alta")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 10. Comparação espaço x frequência

A comparação com a DIP-06 é conceitual. O filtro de média espacial não é equivalente ao passa-baixa ideal, e Sobel não é equivalente ao passa-alta ideal. Observamos apenas a associação geral: suavização privilegia baixas frequências, enquanto detalhes e bordas estão associados a frequências mais altas.

In [ ]:
spatial_mean = spatial_filters.mean_filter(real_image, kernel_size=3)
spatial_edges = features.sobel(real_image)

figure, axes = plt.subplots(2, 3, figsize=(12, 8))
comparisons = [
    (real_image, "Original"),
    (np.clip(spatial_mean, 0, 255), "Média espacial"),
    (np.clip(real_low, 0, 255), "Passa-baixa ideal"),
    (real_image, "Original"),
    (spatial_edges, "Sobel (espaço)"),
    (np.abs(real_high), "|Passa-alta ideal|"),
]
for axis, (result, title) in zip(axes.ravel(), comparisons, strict=True):
    axis.imshow(result, cmap="gray")
    axis.set_title(title)
    axis.axis("off")
figure.tight_layout()
plt.show()

## 11. Exercício final

1. Teste três valores de cutoff e compare as reconstruções.
2. Verifique numericamente que as duas máscaras somam 1.
3. Compare os resultados para uma imagem sintética com transições suaves.
4. Explique por que a visualização logarítmica não substitui a magnitude numérica.

In [ ]:
# TODO: experimente novos cutoffs e registre suas conclusões.
exercise_cutoffs = [5.0, 15.0, 30.0]